# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (from `w02_ml_task_framing.ipynb`) — rank content pages so a reviewer with capacity for ~50 pages/cycle works the highest-priority ones first. This notebook writes that lane's data contract against the full warehouse release, proves three facts about it with real queries on a mid-panel month (`month=2026-03`), builds a first five-feature frame, and performs the leakage trap on purpose.

Sealed test month: **2026-06** (the `_sample` table). Nothing below touches it.

## 0. Setup (Colab or local)

Clones the repo (Colab) or finds the repo root (local), installs DuckDB, and connects to the hosted warehouse release. Run this first.

In [ ]:
import os, sys, subprocess, getpass

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vikraamkumar-ds/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

%pip -q install duckdb huggingface_hub

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell -- this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel dev month -- NEVER the sealed final month (2026-06 / _sample)

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    # partitioned table -- point at ONE month folder while iterating, never the full **/*.parquet glob
    "fact_month": f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>12,} rows")


## 1. The contract, in plain words

**1. What one row means for my lane.** The base table `fact_content_daily_performance` has grain content item x client x calendar day (one row = one page, for one client, on one date). My lane doesn't model at that daily grain, though -- it rolls a month of those daily rows up to **one row per content item (`content_hash_id` within a `client_hash_id`)**, summarizing that item's activity over the window, because the reviewer ranks *pages*, not page-days.

**2. Which table(s).** `fact_content_daily_performance` (the daily fact, partitioned by `month=YYYY-MM`) is the main table. I join `dim_clients` for per-client history coverage (`gsc_data_start`, `ga4_data_start`) so I don't mistake "tracking hadn't started yet" for "no traffic." `dim_content` is available for content metadata but isn't needed for the checks below.

**3. Time window.** I develop on **`month=2026-03`** (one mid-panel month) for every query and feature below. The panel's final month, **2026-06** (also shipped separately as `fact_content_daily_performance_sample`), is the sealed test month -- I never touch it here, because it's the natural outcome window for any past-to-future decline label I'd eventually build.

**4. What I'd predict or rank (label / proxy).** A **proxy** for decline: within the month, whether a page's second-half search visibility fell meaningfully short of its first-half visibility. This is the same style of proxy as the `is_declining_label` from `w02` (rule-defined, not an observed future outcome) -- future work moves to a real forward-window label (trailing 90 days -> next 30 days), which is exactly why the trap in section 3 matters: a proxy built carelessly from the same window it's evaluated on is leakage waiting to happen.

**5. One thing I deliberately exclude.** FlyRank's own product decision fields -- `health_score`, `priority_score`, `needs_ctr_fix`, `is_quick_win`, `action_type` -- are excluded on purpose. They're outputs of an existing hand-written rule, not raw evidence; using them as features would mean the model partly learns to reproduce the rule it's supposed to be compared against (circular, and it's the same trap the `flyrank-context` skill calls out).

In [ ]:
# Schema peek to ground the contract above -- not one of the three required checks, just orientation.
con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_month']}").df()


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated over the month, trailing-half only where noted) | Observed search measurements, logged as they happen -- knowable at the decision moment |
| **Feature (conditional)** | GA4 engagement columns, only where `ga4_data_available IS TRUE` | Real when present, but zero-filled before a client's `ga4_data_start` -- unconditional use would encode "no tracking yet" as "no engagement" |
| **Label / proxy** | the decline flag computed by comparing second-half vs first-half `gsc_impressions` within the month | The thing I'm predicting -- never allowed as a feature, and neither is anything it's directly computed from |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Grouping, joining, splitting, filtering by history window -- never something the model learns a weight for |
| **Excluded** | `health_score`, `priority_score`, `needs_ctr_fix`, `is_quick_win`, `action_type` (not present in this release, named for completeness); raw query/URL/title hashes not needed for this lane | Product-decision outputs (circular use) or private raw-origin context with no role in scoring |

In [ ]:
# no code needed here -- the table above is the answer; move to verification
pass


## 3. Verify it with queries (grain, counts, availability) -- then five features and the trap

Three checks, each with its own query, on `month=2026-03`.

### Query 1 -- grain: one row really is one page x one client x one day

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_month']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"duplicate-grain rows found: {len(grain_check)} (0 confirms the grain holds)")
grain_check


### Query 2 -- my lane's slice: row count and date span

My lane's slice is visible pages only (`gsc_impressions > 0`) -- pages with zero search visibility aren't candidates for a decline-priority review in the first place.

In [ ]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                         AS row_count,
        COUNT(DISTINCT content_hash_id)   AS n_content_items,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date
    FROM {TABLES['fact_month']}
    WHERE gsc_impressions > 0
""").df()

slice_stats


### Query 3 -- availability: filter with `IS TRUE`, count survivors

In [ ]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_month']}
    WHERE gsc_impressions > 0
""").df()

avail["ga4_available_pct"] = 100 * avail["ga4_available_rows"] / avail["total_rows"]
avail


### Five features (max) -- one row per content item, built from this same month

1. **`imp_sum`** -- total `gsc_impressions` over the month. *Knowable at the decision moment because* impressions are logged the day they happen; by month-end every day's count is already in hand.
2. **`clk_sum`** -- total `gsc_clicks` over the month. *Knowable because* clicks are logged the same way, same day.
3. **`avg_position_mean`** -- mean `gsc_avg_position` across days with a real position (position 0 means "no data", per the data dictionary, so those days are excluded from the average). *Knowable because* rank is observed daily, not something that depends on a future outcome.
4. **`ctr_calc`** -- `clk_sum / imp_sum`. *Knowable because* it's arithmetic on two features that are themselves already knowable -- no new information beyond what's in 1 and 2.
5. **`active_days`** -- count of distinct days in the month with `gsc_impressions > 0`. *Knowable because* it's a same-day coverage count, not a forward-looking measurement.

In [ ]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                                      AS imp_sum,
        SUM(gsc_clicks)                                                           AS clk_sum,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)             AS avg_position_mean,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)        AS active_days,
        -- kept separately for the trap below -- NOT one of the five features
        SUM(CASE WHEN report_date <= (SELECT MAX(report_date) FROM {TABLES['fact_month']}) - INTERVAL 15 DAY
                 THEN gsc_impressions ELSE 0 END)                                 AS imp_first_half,
        SUM(CASE WHEN report_date >  (SELECT MAX(report_date) FROM {TABLES['fact_month']}) - INTERVAL 15 DAY
                 THEN gsc_impressions ELSE 0 END)                                 AS imp_second_half
    FROM {TABLES['fact_month']}
    WHERE gsc_impressions > 0
    GROUP BY 1, 2
    HAVING imp_first_half >= 20   -- minimum volume so the half-vs-half comparison isn't pure noise
""").df()

feature_frame["ctr_calc"] = feature_frame["clk_sum"] / feature_frame["imp_sum"]
print(f"{len(feature_frame):,} content items with enough volume this month")
feature_frame.head()


### The trap -- add one label-derived column on purpose, watch the score jump, then remove it

Label (proxy): a page is **declining** this month if its second-half impressions dropped more than 20% versus its first half. `imp_second_half` is literally the quantity the label is computed from -- if I leave it in the feature set, the model doesn't have to learn anything; it just reads the answer off the label's own ingredient.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

data = feature_frame.copy()
data["is_declining"] = (data["imp_second_half"] < 0.8 * data["imp_first_half"]).astype(int)
print("label balance:\n", data["is_declining"].value_counts(normalize=True))

honest_features = ["imp_sum", "clk_sum", "avg_position_mean", "ctr_calc", "active_days"]
leaky_features  = honest_features + ["imp_second_half"]  # the deliberate leak

def quick_auc(cols):
    d = data.dropna(subset=cols + ["is_declining"])
    X, y = d[cols], d["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_features)
leaky_auc  = quick_auc(leaky_features)

print(f"HONEST features only     -> ROC-AUC = {honest_auc:.3f}")
print(f"WITH imp_second_half     -> ROC-AUC = {leaky_auc:.3f}   <-- leakage: the label's own ingredient")


In [ ]:
# Delete the leaky column and keep only the honest number.
data = data.drop(columns=["imp_second_half", "imp_first_half"])
print(f"Honest score kept for this contract: ROC-AUC = {honest_auc:.3f}")
print("imp_second_half / imp_first_half removed from the working frame -- confirmed:", 
      not {"imp_second_half", "imp_first_half"}.issubset(set(data.columns)))


## 4. Data limits

**Named limitation: this month-internal proxy can't tell real decline from a mid-month blip, and it inherits the panel's unbalanced history.** Comparing first-half vs second-half `gsc_impressions` within one calendar month catches short-term swings (a holiday week, a SERP feature that came and went) just as easily as genuine multi-month decay -- it has no way to tell the two apart from 30 days alone. On top of that, `dim_clients.gsc_data_start` / `ga4_data_start` differ per client, so `active_days` and `avg_position_mean` mean something slightly different for a client with a full month of history than for one whose tracking started mid-month; and GA4 fields are zero-filled with `ga4_data_available = FALSE` before a client's GA4 start, which Query 3 above exists specifically to catch (a blind average over that column would read as "low engagement" instead of "not tracked yet"). The fix for the first issue is the planned move to a real forward-window label (trailing 90 days -> next 30 days, evaluated only on the sealed 2026-06 month) once this contract graduates past the proxy stage.

In [ ]:
# Supporting check for the limitation above: how much history variation exists across clients touched by this slice.
history_spread = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_start, MAX(gsc_data_start) AS latest_start,
           COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
    WHERE client_hash_id IN (SELECT DISTINCT client_hash_id FROM {TABLES['fact_month']} WHERE gsc_impressions > 0)
""").df()
history_spread


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.